In [1]:
# Cell 1: imports + tiny helpers
import pandas as pd

from rl.train import train
from rl.evaluate import evaluate_custom_agent
from rl.visualize import plot_training_curve
from rl.extension_configs import TRAIN_VARIANTS, build_config

TRAIN_VARIANTS_TO_RUN = ["baseline_shared", "safe_shared"]
EVAL_TRAFFIC = ["shared", "dense"]   # later you can switch dense -> stress
EVAL_SEEDS = list(range(1000, 1010))


def make_train_kwargs(variant_name):
    variant = TRAIN_VARIANTS[variant_name]
    configs = [
        build_config(traffic_name, variant["reward_name"])
        for traffic_name in variant["traffic_names"]
    ]
    if len(configs) == 1:
        return {"env_config": configs[0]}
    return {"train_env_configs": configs}


def make_eval_config(traffic_name):
    return build_config(traffic_name, "baseline")


/Users/tomas/Documents/Cours 2025-2026/Current/RL/Projet/.venv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [ ]:
# Cell 2: train the two agents
agents = {}
metrics_by_variant = {}

for variant_name in TRAIN_VARIANTS_TO_RUN:
    agent, metrics = train(
        seed=0,
        run_dir=f"artifacts/extension/{variant_name}/seed_0",
        **make_train_kwargs(variant_name),
    )
    agents[variant_name] = agent
    metrics_by_variant[variant_name] = metrics


Using device: mps
step=23, return=16.43, epsilon=1.000
step=51, return=20.09, epsilon=1.000
step=81, return=21.40, epsilon=1.000
step=92, return=8.60, epsilon=1.000
step=116, return=17.98, epsilon=1.000
step=119, return=1.77, epsilon=1.000
step=128, return=7.20, epsilon=1.000
step=135, return=5.76, epsilon=1.000
step=149, return=9.53, epsilon=1.000
step=160, return=8.25, epsilon=1.000
step=167, return=5.75, epsilon=1.000
step=188, return=15.15, epsilon=1.000
step=198, return=7.88, epsilon=1.000
step=221, return=17.66, epsilon=1.000
step=225, return=2.40, epsilon=1.000
step=229, return=2.76, epsilon=1.000
step=241, return=8.68, epsilon=1.000
step=247, return=3.89, epsilon=1.000
step=255, return=5.69, epsilon=1.000
step=273, return=14.29, epsilon=1.000
step=279, return=4.01, epsilon=1.000
step=288, return=7.32, epsilon=1.000
step=315, return=19.77, epsilon=1.000
step=320, return=3.60, epsilon=1.000
step=330, return=8.56, epsilon=1.000
step=339, return=7.75, epsilon=1.000
step=347, return

In [ ]:
# Cell 3: quick look at the training curves
for variant_name in TRAIN_VARIANTS_TO_RUN:
    plot_training_curve(metrics_by_variant[variant_name], label=variant_name)


Using device: mps
3 45 1.0
step=22, return=17.82, epsilon=1.000
3 45 1.0
step=27, return=3.46, epsilon=1.000
4 45 1.0
step=32, return=3.84, epsilon=1.000
4 30 0.7


KeyboardInterrupt: 

In [ ]:
# Cell 4: evaluate both agents on two environments
rows = []

for variant_name, agent in agents.items():
    for traffic_name in EVAL_TRAFFIC:
        result = evaluate_custom_agent(
            agent,
            seeds=EVAL_SEEDS,
            env_config=make_eval_config(traffic_name),
        )

        rows.append(
            {
                "variant": variant_name,
                "eval_env": traffic_name,
                "mean_return": result["mean_return"],
                "std_return": result["std_return"],
                "mean_length": result["mean_length"],
                "crash_rate": result["crash_rate"],
                "mean_speed": result["mean_speed"],
                "lane_change_rate": result["lane_change_rate"],
            }
        )

results_df = pd.DataFrame(rows)
display(results_df.round(3))


NameError: name 'agent' is not defined

In [ ]:
# Cell 5: cleaner comparison table
comparison = results_df.pivot(
    index="variant",
    columns="eval_env",
    values=["mean_return", "crash_rate", "mean_speed", "lane_change_rate"],
)

display(comparison.round(3))
